In [8]:
import tkinter as tk
from pathlib import Path
from tkinter import filedialog
import numpy as np
import pandas as pd
from IPython import get_ipython

get_ipython().run_line_magic("gui", "tk")
root = tk.Tk()
root.withdraw()
root.call("wm", "attributes", ".", "-topmost", True)


########## Richard's code to combine cell count CSVs ###################

def prepare_df(directory: Path) -> pd.DataFrame:
    df = pd.concat(
        [
            pd.read_csv(directory / file).assign(file=file.name)
            for file in directory.iterdir()
            if file.suffix == ".csv" and not file.name == "output.csv"
        ]
    ).rename(columns={" ": "cell"})

    df["file"] = df["file"].astype("category")
    u = df.columns.tolist()
    return df[[u[-1]] + u[:-1]]


directory = Path(filedialog.askdirectory(title="Choose folder containing csvs with the UNET counts"))

axis = ""

if axis == "":
    segment = 0
else:
    segment = int(input("Enter the number of splits."))


df = prepare_df(directory)

if segment == 0:
    df["layer"] = 0
else:
    vals = df[axis] - df[axis].min()
    df["layer"] = (vals // ((np.ptp(df[axis]) + 0.1) / segment)).astype(int).astype("category")

df = df.groupby(["file", "layer"]).size().reset_index()
df = df.pivot(index="file", columns="layer", values=0)

################## Sarah Johnson's code to add area measurement ####################


df = df.rename(columns={0:"Cells"})
df["Area (um^2)"] = 0

df["Cells per 40,000 um^2"] = 0
df["Cells per 40,000 um^2"] = df["Cells per 40,000 um^2"].astype(float)


#IMPORTANT NOTE: The area measurements from FIJI are in pixels, because the pixel to micron ratio for x and y was made 1:1  
#during the image processing. The original pixel to micron ratio for x and y, which should always be the same for the
#Olympus 30x objective, was 1 pixel = 0.82864 microns. I multiply the area measurements acquired by FIJI by (0.82864)^2
#to convert pixels squared to microns squared.

def add_area(k,l): 
    micron_area = 1760000*0.82864*0.82864
    df.loc[k, "Area (um^2)"] = micron_area
    numb = (l.Cells/micron_area)*40000
    df.loc[k, "Cells per 40,000 um^2"] = round(numb,0)


for k,l in df.iterrows():
    add_area(k,l)


df.to_csv(directory / "output.csv")

print("Done!")








Done!
